In [ ]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    print("Colab detected: ensuring compatible transformers version...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "-q", "transformers>=4.30.0"])

import random
import re
import unicodedata
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

from transformers import (
    AutoTokenizer,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

from pathlib import Path

PROJECT_DIR = Path("/content")
DATA_PATH = PROJECT_DIR / "jigsaw-unintended-bias-train.csv"
PART1_MODEL_DIR = PROJECT_DIR / "saved_model/part1_baseline"

print("Data exists:", DATA_PATH.exists())
print("Part1 model exists:", PART1_MODEL_DIR.exists())

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running in Colab:", IN_COLAB)
print("Project directory:", PROJECT_DIR.resolve())
print("Device:", DEVICE)
print("Data exists:", DATA_PATH.exists())
print("Part1 model exists:", PART1_MODEL_DIR.exists())

Colab detected: ensuring compatible transformers version...
Data exists: True
Part1 model exists: True
Running in Colab: True
Project directory: /content
Device: cuda
Data exists: True
Part1 model exists: True


In [21]:
assert DATA_PATH.exists(), f"Missing dataset file: {DATA_PATH}"

df = pd.read_csv(DATA_PATH, usecols=["comment_text", "toxic"], low_memory=False)
df = df.dropna(subset=["comment_text", "toxic"]).copy()
df["label"] = (df["toxic"] >= 0.5).astype(int)

TARGET_TOTAL = 120_000
TRAIN_SIZE = 100_000
EVAL_SIZE = 20_000

subset_df, _ = train_test_split(
    df,
    train_size=TARGET_TOTAL,
    stratify=df["label"],
    random_state=SEED,
)

train_df, eval_df = train_test_split(
    subset_df,
    train_size=TRAIN_SIZE,
    stratify=subset_df["label"],
    random_state=SEED,
)

assert len(train_df) == TRAIN_SIZE
assert len(eval_df) == EVAL_SIZE

print("Train shape:", train_df.shape)
print("Eval shape:", eval_df.shape)
print("Train toxic ratio:", train_df['label'].mean())
print("Eval toxic ratio:", eval_df['label'].mean())

Train shape: (100000, 3)
Eval shape: (20000, 3)
Train toxic ratio: 0.07997
Eval toxic ratio: 0.07995


In [22]:
MAX_LEN = 128
BATCH_SIZE = 64
MODEL_NAME = "distilbert-base-uncased"

def load_baseline_model_and_tokenizer():
    if PART1_MODEL_DIR.exists():
        tokenizer = AutoTokenizer.from_pretrained(str(PART1_MODEL_DIR))
        model = DistilBertForSequenceClassification.from_pretrained(str(PART1_MODEL_DIR))
        source = str(PART1_MODEL_DIR)
    else:
        tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
        model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
        source = MODEL_NAME
    model.to(DEVICE)
    model.eval()
    return model, tokenizer, source

baseline_model, baseline_tokenizer, baseline_source = load_baseline_model_and_tokenizer()
print("Loaded baseline from:", baseline_source)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loaded baseline from: /content/saved_model/part1_baseline


In [23]:
def predict_probs(texts, model, tokenizer, batch_size=BATCH_SIZE, max_len=MAX_LEN):
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            enc = tokenizer(
                batch_texts,
                truncation=True,
                padding=True,
                max_length=max_len,
                return_tensors="pt",
            )
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            all_probs.extend(probs)
    return np.array(all_probs)

def compute_basic_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    fnr = fn / (fn + tp) if (fn + tp) else np.nan
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro"),
        "fnr": fnr,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)
    }

In [ ]:
ZWSP = "\u200b"
HOMOGLYPHS = {
    "a": "а",
    "e": "е",
    "o": "о",
    "A": "А",
    "E": "Е",
    "O": "О",
}

TOXIC_HINTS = {
    "hate", "idiot", "stupid", "dumb", "kill", "moron", "trash",
    "shut", "loser", "ugly", "toxic", "racist", "nazi", "terrorist"
}

def insert_zwsp(word, rng):
    if len(word) < 4:
        return word
    out = []
    i = 0
    while i < len(word):
        step = rng.choice([2, 3])
        out.append(word[i:i+step])
        i += step
        if i < len(word):
            out.append(ZWSP)
    return "".join(out)

def substitute_homoglyphs(word):
    return "".join(HOMOGLYPHS.get(ch, ch) for ch in word)

def duplicate_chars(word, rng, p=0.2):
    out = []
    for ch in word:
        out.append(ch)
        if ch.isalpha() and rng.random() < p:
            out.append(ch)
    return "".join(out)

def perturb(text, seed_offset=0):
    rng = random.Random(SEED + seed_offset)
    tokens = text.split()
    perturbed_tokens = []

    for t in tokens:
        base = re.sub(r"[^A-Za-z]", "", t).lower()
        w = t

        if base in TOXIC_HINTS or any(h in base for h in ["hate", "kill", "idiot", "stupid"]):
            w = insert_zwsp(w, rng)


        w = substitute_homoglyphs(w)

        w = duplicate_chars(w, rng, p=0.2)

        perturbed_tokens.append(w)

    return " ".join(perturbed_tokens)

sample_text = "you are a stupid idiot and i hate you"
print("Original:", sample_text)
print("Perturbed:", perturb(sample_text, seed_offset=1))

Original: you are a stupid idiot and i hate you
Perturbed: yyоuu аrе а stu​pi​d idd​iо​t аnd i hhаа​ttе yyоu


In [25]:
eval_texts = eval_df["comment_text"].tolist()
clean_probs = predict_probs(eval_texts, baseline_model, baseline_tokenizer)
clean_pred = (clean_probs >= 0.4).astype(int)

cand_idx = np.where((clean_pred == 1) & (clean_probs >= 0.7))[0]
print("Candidate toxic comments (pred=1, conf>=0.7):", len(cand_idx))

N_ATTACK = min(500, len(cand_idx))
rng = np.random.default_rng(SEED)
sel_idx = rng.choice(cand_idx, size=N_ATTACK, replace=False) if N_ATTACK > 0 else np.array([], dtype=int)

attack_df = eval_df.iloc[sel_idx].copy()
attack_df["clean_prob"] = clean_probs[sel_idx] if N_ATTACK > 0 else []
attack_df["perturbed_text"] = [perturb(t, seed_offset=i) for i, t in enumerate(attack_df["comment_text"].tolist())]

if N_ATTACK > 0:
    pert_probs = predict_probs(attack_df["perturbed_text"].tolist(), baseline_model, baseline_tokenizer)
    pert_pred = (pert_probs >= 0.4).astype(int)

    attack_df["pert_prob"] = pert_probs
    attack_df["pert_pred"] = pert_pred

    asr = ((attack_df["pert_pred"] == 0).sum() / N_ATTACK)
    avg_clean_conf = float(attack_df["clean_prob"].mean())
    avg_pert_conf = float(attack_df["pert_prob"].mean())
else:
    asr = np.nan
    avg_clean_conf = np.nan
    avg_pert_conf = np.nan

asr_table = pd.DataFrame([
    {
        "n_attacked": int(N_ATTACK),
        "attack_success_rate": asr,
        "avg_conf_before": avg_clean_conf,
        "avg_conf_after": avg_pert_conf,
    }
])

display(asr_table)

if N_ATTACK > 0:
    print("Example original:")
    print(attack_df.iloc[0]["comment_text"][:400])
    print("\nExample perturbed:")
    print(attack_df.iloc[0]["perturbed_text"][:400])

Candidate toxic comments (pred=1, conf>=0.7): 941


,n_attacked,attack_success_rate,avg_conf_before,avg_conf_after
0,500,0.96,0.897721,0.039866


Example original:
All stupid, no brains, is a more accurate description of Trudeau.

Example perturbed:
Аlll stt​uup​id​, nо brаinss, is а mооrе аcccurrаttе dеsccrriptiоn оf TTrudеааu.


In [ ]:
poison_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ToxicityDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(label, dtype=torch.long)
        return item

poison_train_df = train_df.copy().reset_index(drop=True)
n_flip = int(0.05 * len(poison_train_df))
flip_idx = np.random.default_rng(SEED).choice(len(poison_train_df), size=n_flip, replace=False)
poison_train_df.loc[flip_idx, "label"] = 1 - poison_train_df.loc[flip_idx, "label"]

print("Poisoned rows:", n_flip)
print("Original train toxic ratio:", train_df['label'].mean())
print("Poisoned train toxic ratio:", poison_train_df['label'].mean())

clean_train_dataset = ToxicityDataset(
    train_df["comment_text"].tolist(),
    train_df["label"].tolist(),
    poison_tokenizer,
    max_len=MAX_LEN,
)

poison_train_dataset = ToxicityDataset(
    poison_train_df["comment_text"].tolist(),
    poison_train_df["label"].tolist(),
    poison_tokenizer,
    max_len=MAX_LEN,
)

clean_eval_dataset = ToxicityDataset(
    eval_df["comment_text"].tolist(),
    eval_df["label"].tolist(),
    poison_tokenizer,
    max_len=MAX_LEN,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Poisoned rows: 5000
Original train toxic ratio: 0.07997
Poisoned train toxic ratio: 0.12261


In [29]:
def trainer_compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]
    preds = (probs >= 0.4).astype(int)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

def train_model(train_dataset, tag):
    model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    out_dir = PROJECT_DIR / f"outputs/part3_{tag}"
    out_dir.mkdir(parents=True, exist_ok=True)

    args = TrainingArguments(
        output_dir=str(out_dir),
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=200,
        report_to="none",
        seed=SEED,
        fp16=torch.cuda.is_available(),
    )

    tr = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=clean_eval_dataset,
        processing_class=poison_tokenizer,
        compute_metrics=trainer_compute_metrics,
    )

    tr.train()

    pred = tr.predict(clean_eval_dataset)
    probs = torch.softmax(torch.tensor(pred.predictions), dim=1).numpy()[:, 1]
    y_pred = (probs >= 0.4).astype(int)
    y_true = pred.label_ids

    m = compute_basic_metrics(y_true, y_pred)
    return tr, m

print("Training clean reference model (for before/after comparison)...")
clean_trainer, clean_metrics = train_model(clean_train_dataset, "clean")

print("Training poisoned model (5% label flip)...")
poison_trainer, poison_metrics = train_model(poison_train_dataset, "poisoned")

Training clean reference model (for before/after comparison)...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.154038,0.135453,0.946850,0.813538
2,0.120037,0.164016,0.947650,0.800216
3,0.075448,0.229775,0.945400,0.806672


Training poisoned model (5% label flip)...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.298679,0.159457,0.945250,0.807468
2,0.257954,0.157237,0.947500,0.799183
3,0.220604,0.185668,0.942650,0.798684


In [30]:
comparison = pd.DataFrame([
    {"model": "clean_retrained", **clean_metrics},
    {"model": "poisoned_5pct", **poison_metrics},
])

display(comparison[["model", "accuracy", "f1_macro", "fnr", "tn", "fp", "fn", "tp"]])

delta = {
    "accuracy_drop": poison_metrics["accuracy"] - clean_metrics["accuracy"],
    "f1_macro_drop": poison_metrics["f1_macro"] - clean_metrics["f1_macro"],
    "fnr_change": poison_metrics["fnr"] - clean_metrics["fnr"],
}

delta_df = pd.DataFrame([delta])
display(delta_df)

print("Interpretation:")
print("Negative accuracy/f1 delta means poisoning hurt quality.")
print("Positive fnr_change means poisoning made model miss more toxic content.")

,model,accuracy,f1_macro,fnr,tn,fp,fn,tp
0,clean_retrained,0.94540,0.806672,0.385241,17925,476,616,983
1,poisoned_5pct,0.94265,0.798684,0.393371,17883,518,629,970


,accuracy_drop,f1_macro_drop,fnr_change
0,-0.00275,-0.007988,0.00813


Interpretation:
Negative accuracy/f1 delta means poisoning hurt quality.
Positive fnr_change means poisoning made model miss more toxic content.


In [ ]:
import shutil
from google.colab import files

model_dir = "/content/saved_model"

zip_path = "/content/part1_baseline.zip"
shutil.make_archive(zip_path.replace(".zip", ""), 'zip', model_dir)

files.download(zip_path)

## Discussion Questions

- Which attack is more operationally dangerous for a live platform, and why?

Ans: The perturbation / evasion attack is more operationally dangerous for a live platform. It is because it had 96% attack success rate and it dropped the model's confidence from about 0.898 to 0.040 which means an attacker can change the text slightly and bypass moderations imediately which is more operationallly dangerous for a live platform.

- Consider: the evasion attack requires per-comment effort from the attacker; the poisoning attack requires access to the training pipeline. Which threat model is more realistic for a social platform, and what does that imply for where defenses should be prioritized? 

Ans: The evasion attack is more realistic for a social platform because it only requires an attacker to change the text of their own comment before posting which matches how users behave on social platforms and poisoning attack is much harder as it needs access to the training pipeline. So the practical priority is to protect the live moderation pipeline against evasion first.